In [1]:
pip install pandas numpy scikit-learn torch openpyxl

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("nba_last_7_days.xlsx")

print(df.head())
print(df.columns)

   personId  period        clock   actionType              subType shotResult  \
0         0       1  PT12M00.00S       period                start        NaN   
1   1629674       1  PT12M00.00S    Jump Ball                  NaN        NaN   
2   1626171       1  PT11M42.00S     Turnover            Lost Ball        NaN   
3   1627759       1  PT11M42.00S          NaN                  NaN        NaN   
4   1627759       1  PT11M32.00S  Missed Shot  Step Back Jump shot     Missed   

   shotDistance  shotValue                               description    gameId  
0             0          0         Start of 1st Period (3:44 PM EST)  22500702  
1             0          0  Jump Ball Queta vs. Turner: Tip to Kuzma  22500702  
2             0          0         Portis Lost Ball Turnover (P1.T1)  22500702  
3             0          0                       Brown STEAL (1 STL)  22500702  
4            27          3    MISS Brown 27' 3PT Step Back Jump Shot  22500702  
Index(['personId', 'period'

In [3]:
# points scored
df["points"] = np.where(df["shotResult"] == "Made", df["shotValue"], 0)

# field goal attempt
df["fga"] = np.where(df["actionType"] == "Shot", 1, 0)

# field goal made
df["fgm"] = np.where(df["shotResult"] == "Made", 1, 0)

# three pointers made
df["three_pm"] = np.where(
    (df["shotResult"] == "Made") & (df["shotValue"] == 3), 1, 0
)

In [4]:
player_games = df.groupby(["personId", "gameId"]).agg({
    "points": "sum",
    "fga": "sum",
    "fgm": "sum",
    "three_pm": "sum"
}).reset_index()

player_games = player_games.sort_values(["personId", "gameId"])

print(player_games.head())

   personId    gameId  points  fga  fgm  three_pm
0         0  22500702       0    0    0         0
1         0  22500703       0    0    0         0
2         0  22500704       0    0    0         0
3         0  22500705       0    0    0         0
4         0  22500706       0    0    0         0


In [5]:
from sklearn.preprocessing import MinMaxScaler

features = ["points","fga","fgm","three_pm"]

scaler = MinMaxScaler()
player_games[features] = scaler.fit_transform(player_games[features])

In [6]:
WINDOW = 3

X = []
y = []

for player in player_games["personId"].unique():

    pdata = player_games[player_games["personId"] == player]
    pdata = pdata[features].values

    if len(pdata) <= WINDOW:
        continue

    for i in range(len(pdata) - WINDOW):
        X.append(pdata[i:i+WINDOW])
        y.append(pdata[i+WINDOW])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (191, 3, 4)
y shape: (191, 4)


In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [8]:
import torch

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [9]:
import torch.nn as nn

class PlayerLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x):

        out, _ = self.lstm(x)

        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [10]:
input_size = len(features)

model = PlayerLSTM(
    input_size=input_size,
    hidden_size=64,
    num_layers=2
)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [11]:
EPOCHS = 30

for epoch in range(EPOCHS):

    model.train()

    outputs = model(X_train)

    loss = criterion(outputs, y_train)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch} Loss {loss.item():.4f}")

Epoch 0 Loss 0.0475
Epoch 5 Loss 0.0346
Epoch 10 Loss 0.0244
Epoch 15 Loss 0.0183
Epoch 20 Loss 0.0186
Epoch 25 Loss 0.0180


In [12]:
model.eval()

with torch.no_grad():

    preds = model(X_test)

    test_loss = criterion(preds, y_test)

print("Test Loss:", test_loss.item())

Test Loss: 0.018741952255368233


In [14]:
sample = X_test[0].unsqueeze(0)

predicted = model(sample)

predicted = predicted.detach().numpy()

predicted = scaler.inverse_transform(predicted)

print(predicted)

[[ 5.740782   -0.01750308  2.5006797   0.8082753 ]]
